# Module 5 Homework - Solutions

Exercises solved based on the material in `05-data-platforms/notes/`.

Each question includes a **command sequence** to reproduce/validate the result.


## Question 1. Bruin Pipeline Structure

- **Correct answer:** `.bruin.yml` and `pipeline/` with `pipeline.yml` and `assets/`
- **Command sequence:**
```bash
# Step 1: initialize the course template
bruin init zoomcamp my-taxi-pipeline

# Step 2: enter the project
cd my-taxi-pipeline

# Step 3: inspect the structure
tree -L 3

# Step 4: confirm required paths
ls -la .bruin.yml pipeline/pipeline.yml pipeline/assets
```
- **Expected result:** `.bruin.yml` exists at root, plus `pipeline/pipeline.yml` and `pipeline/assets/`.


## Question 2. Materialization Strategies

- **Correct answer:** `time_interval` - incremental based on a time column
- **Command sequence:**
```bash
# Step 1: locate reference in module material
rg -n "strategy: time_interval" 05-data-platforms/notes/03-nyc-taxi-pipeline.md

# Step 2: run one time window
bruin run ./pipeline/pipeline.yml --start-date 2022-01-01 --end-date 2022-02-01

# Step 3: verify data loaded in that interval
bruin query --connection duckdb-default --query "SELECT MIN(pickup_datetime), MAX(pickup_datetime), COUNT(*) FROM staging.trips"
```
- **Expected result:** strategy reprocesses by time window (delete + insert for the interval).


## Question 3. Pipeline Variables

- **Correct answer:** `bruin run --var 'taxi_types=[\"yellow\"]'`
- **Command sequence:**
```bash
# Step 1: run pipeline overriding array variable
bruin run ./pipeline/pipeline.yml --var 'taxi_types=["yellow"]'

# Step 2: verify only yellow was processed
bruin query --connection duckdb-default --query "SELECT DISTINCT taxi_type FROM ingestion.trips ORDER BY 1"
```
- **Expected result:** only `yellow` appears in taxi types output.


## Question 4. Running with Dependencies

- **Correct answer:** `bruin run ingestion/trips.py --downstream`
- **Command sequence (recommended explicit form):**
```bash
# Step 1: validate pipeline
bruin validate ./pipeline/pipeline.yml

# Step 2: run changed asset + all downstream
bruin run ./pipeline/pipeline.yml --asset ingestion.trips --downstream

# Step 3: visualize asset dependencies
bruin lineage ./pipeline/pipeline.yml --asset ingestion.trips
```
- **Expected result:** `ingestion.trips` runs first, followed by dependent `staging` and `reports` assets.


## Question 5. Quality Checks

- **Correct answer:** `name: not_null`
- **Command sequence:**
```bash
# Step 1: confirm check in module material
rg -n "name: not_null" 05-data-platforms/notes/03-nyc-taxi-pipeline.md

# Step 2: validate pipeline definitions
bruin validate ./pipeline/pipeline.yml

# Step 3: run asset with quality checks
bruin run ./pipeline/pipeline.yml --asset staging.trips

# Step 4: verify via SQL there are no nulls
bruin query --connection duckdb-default --query "SELECT COUNT(*) AS null_pickups FROM staging.trips WHERE pickup_datetime IS NULL"
```
- **Expected result:** `null_pickups = 0`.


## Question 6. Lineage and Dependencies

- **Correct answer:** `bruin lineage`
- **Command sequence:**
```bash
# Step 1: visualize full pipeline lineage
bruin lineage ./pipeline/pipeline.yml

# Step 2: visualize lineage for a specific asset
bruin lineage ./pipeline/pipeline.yml --asset ingestion.trips
```
- **Expected result:** graph with dependency relationships between assets.


## Question 7. First-Time Run

- **Correct answer:** `--full-refresh`
- **Command sequence:**
```bash
# Step 1: first run with full rebuild
bruin run ./pipeline/pipeline.yml --full-refresh

# Step 2: verify tables were recreated and populated
bruin query --connection duckdb-default --query "SELECT COUNT(*) FROM ingestion.trips"
bruin query --connection duckdb-default --query "SELECT COUNT(*) FROM staging.trips"
bruin query --connection duckdb-default --query "SELECT COUNT(*) FROM reports.trips_report"
```
- **Expected result:** tables created from scratch with counts > 0 (depending on processed window).


## Final Summary

1. `.bruin.yml` + `pipeline/` + `pipeline.yml` + `assets/`
2. `time_interval`
3. `bruin run --var 'taxi_types=[\"yellow\"]'`
4. `bruin run ingestion/trips.py --downstream`
5. `name: not_null`
6. `bruin lineage`
7. `--full-refresh`
